In [ ]:
!pip install pandas
!pip install requests
!pip install selenium
!pip install beautifulsoup4
!pip install lxml
!pip install openpyxl

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import re



In [ ]:
s=Service(r"C:\Users\harsh\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe")
driver= webdriver.Chrome(service= s)

In [ ]:
driver.get("https://www.makemytrip.com/hotels/hotel-listing/?checkin=11202024&checkout=11212024&locusId=CTHYDERA&locusType=city&city=CTHYDERA&country=IN&searchText=Hyderabad&roomStayQualifier=2e0e&_uCurrency=INR&reference=hotel&type=city&rsc=1e2e0e")

In [ ]:
time.sleep(1)

user_input=driver.find_element(by=By.XPATH , value='//*[@id="MERGE_PROPERTY_TYPE"]/ul/li[1]/span[1]/label/div/span')

user_input.click()
time.sleep(2)

In [ ]:
old_height=driver.execute_script('return document.body.scrollHeight')

while True:

    driver.execute_script('window.scrollTo(0,document.body.scrollHeight)')
    time.sleep(3)
    new_height=driver.execute_script('return document.body.scrollHeight')


    if(old_height==new_height):
        break
    else:
        old_height=new_height



In [ ]:
html = driver.page_source

with open('MMTBengaluru.html', 'w', encoding='utf-8') as file:
    file.write(html)
    file.close()


In [ ]:
with open('MMTBengaluru.html','r') as file:
    html=file.read()
    file.close()

In [ ]:
print(html)

In [ ]:
soup=BeautifulSoup(html,'html.parser')
soup.prettify()

In [ ]:
containers= soup.find_all('div',{'class':'flexOne makeFlex'})

In [ ]:
names=[]
price=[]
taxes=[]
rating=[]
reviewCount=[]
city=[]
distanceAirport=[]
freeCancel=[]
CoupleFriendly=[]

In [ ]:

for i in containers:
    names.append(i.find('span', {'class': 'wordBreak appendRight10'}).text)  ## Names 
    price.append(i.find('p', {'class': 'priceText latoBlack font22 blackText appendBottom5'}).text)  ## Prices
    rating.append(i.find('span', {'class': 'latoBlack blueBg ratingWrapper font14 appendLeft8 rating'}).text  if i.find('span', {'class': 'latoBlack blueBg ratingWrapper font14 appendLeft8 rating'}) else np.nan )   ## ratings
    taxes.append(i.find('p', {'class': 'font14 midGreyText'}).text)  ##taxes
    reviewCount.append(i.find('span',{'itemprop':'reviewCount'}).text  if i.find('span',{'itemprop':'reviewCount'}) else np.nan)
    city.append(i.find('div', {'class': 'addrContainer'}).find('span', {'class': 'blueText'}).text)  ## near places
    distanceAirport.append(i.find('span', {'class': 'latoRegular'}).text   if i.find('span', {'class': 'latoRegular'}) else "Not Provided" )  ## distance from airport/beach
    freeCancel.append(i.find('div', {'class': "persuasion__item greenText font14 pc__inclusionPerNew"}).text if i.find('div', {'class': "persuasion__item greenText font14 pc__inclusionPerNew"}) else np.nan)  ## Free Cancelation
    CoupleFriendly.append(i.find('div', {'class': "persuasion__item pc__hotelCategoryPerNew"}).text if i.find('div', {'class': "persuasion__item pc__hotelCategoryPerNew"}) else np.nan) ## couple friendly



In [ ]:
df=pd.DataFrame({
    'Name':names,
    'City':city,
    'Rating':rating,
    'Review Count':reviewCount,
    'Price':price,
    'Tax':taxes,
    'Distance From Nearby Routes':distanceAirport,
    'Free Cancelation':freeCancel,
    'Couple Friendly':CoupleFriendly
})

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.to_csv('Bengaluru_hotels.csv', index=False)

In [ ]:
df.to_excel('bengaluru_hotels.xlsx', index=False)

## Cleanning Process

<h4>Mistakes<h4>

1. Rating,reveiew count, -  Rating has nan values

2. price-  convert into int format 

3. Tax - tax also in int format and generate a total amount 

4. Distance - there are so unusual data , remove that 

5. Free Cancellation,couple friendly - keep in yes or no 

In [ ]:
df=pd.read_csv('Bengaluru_hotels.csv')

In [ ]:
df.info()

In [ ]:
df['Name'].duplicated().sum()

In [ ]:
df['City'].value_counts()

In [ ]:
df['Couple Friendly'].isna().sum()

In [ ]:
df1=df

In [ ]:
df1['Couple Friendly'].fillna('No',inplace=True)

In [ ]:
df1['Couple Friendly']=df1['Couple Friendly'].apply(lambda x: True if x=='Couple Friendly' else False)

In [ ]:
df1['Free Cancelation'].fillna('No',inplace=True)
df1['Free Cancelation']=df1['Free Cancelation'].str.contains('Free',case=False)

In [ ]:
df1['Price']

In [ ]:
df1['Price'] = df1['Price'].str.replace(r'₹\s?|,', '', regex=True).astype(int)

In [ ]:
df1['Tax'] = df1['Tax'].str.extract(r'(\d+)').astype(int)

In [ ]:
df1['Tax']

In [ ]:
df1['Total Amount']=df1['Price']+df1['Tax']

In [ ]:
df1

In [ ]:
df1['Distance From Nearby Routes']=df1['Distance From Nearby Routes'].apply(lambda x: x if any(char.isdigit() for char in x) else 'Not provided')

In [ ]:
df1

In [ ]:
df1[df1['Distance From Nearby Routes']!="Not provided"]

In [ ]:
df1[df1['Rating'].isna()]

In [ ]:
df1.to_csv('Bengaluru_hotels.csv', index=False)